In [1]:
!pip install pandas==2.1.4 -qq
!pip install google-cloud-bigquery==3.40.0 -qq
!pip install openpyxl==3.1.2 -qq
!pip install db-dtypes==1.5.0 -qq
!pip install catboost -qq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow-skinny 2.18.0 requires protobuf<6,>=3.12.0, but you have protobuf 6.33.6 which is incompatible.


In [2]:
import calendar
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from datetime import datetime
from google.cloud import bigquery
from zoneinfo import ZoneInfo
from matplotlib import pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [3]:
def generate_month_query(table_full_name: str, tz_name: str = "Asia/Ho_Chi_Minh", ref_date: datetime | None = None) -> str:
    """
    Trả về query cho "tháng hiện tại" theo timezone tz_name.
    table_full_name ví dụ: "real-estate-project-445516.real_estate_data.ads_data"
    ref_date dùng để test (nếu None thì lấy thời điểm hiện tại).
    """
    tz = ZoneInfo(tz_name)
    now = ref_date.astimezone(tz) if ref_date else datetime.now(tz)
    
    # year = now.year
    # month = now.month
    year = 2026
    month = 3

    # ngày bắt đầu tháng (00:00:00) và ngày cuối tháng (00:00:00)
    # first_day_str = f"{year:04d}-{month:02d}-01T00:00:00"
    # last_day_num = calendar.monthrange(year, month)[1]
    # last_day_str = f"{year:04d}-{month:02d}-{last_day_num:02d}T00:00:00"

    # query = f"""select *
    #             from `{table_full_name}`
    #             where (`Ngày đăng` <= '{first_day_str}' and `Ngày hết hạn` >= '{first_day_str}') or
    #                   (`Ngày đăng` >= '{first_day_str}' and `Ngày đăng` <= '{last_day_str}');"""

    first_day_str = f"{year:04d}-{month:02d}-01T00:00:00"
    last_day_num = calendar.monthrange(year, month)[1]
    last_day_str = f"{year:04d}-{month:02d}-{last_day_num:02d}T00:00:00"

    query = f"""select *
                from `{table_full_name}`
                where (`Ngày gia hạn` <= '{last_day_str}' and `Ngày gia hạn` >= '{first_day_str}');"""
    
    return query


In [4]:
client = bigquery.Client.from_service_account_json("/lakehouse/default/Files/real-estate-project-445516-83dc50b692bc.json")

In [5]:
table = "real-estate-project-445516.real_estate_data.ads_data"
query = generate_month_query(table)

In [6]:
# df = client.query(query).to_dataframe()

In [7]:
df = pd.read_csv("/lakehouse/default/Files/ads_data_2026_03.csv")
# df.to_csv("/lakehouse/default/Files/ads_data_2026_03.csv")

/tmp/ipykernel_433/389501649.py:1: DtypeWarning: Columns (10,29,30,31,32,33,34) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/lakehouse/default/Files/ads_data_2026_03.csv")


In [8]:
df.columns

Index(['Unnamed: 0', 'Mã tin', 'Ngày đăng', 'Ngày hết hạn', 'Loại tin',
       'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1',
       'Địa chỉ 2', 'Khoảng giá', 'Diện tích', 'Link dự án', 'Tên dự án',
       'Tiêu đề', 'Mô tả', 'Số tầng lookup', 'Số tầng', 'Căn góc',
       'Ngày gia hạn', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Hướng nhà',
       'Hướng ban công', 'Mặt tiền', 'Đường vào', 'Pháp lý', 'Nội thất',
       'Thời gian dự kiến vào ở', 'Mức giá nước', 'Mức giá internet',
       'Tiện ích', 'Mức giá điện', 'Phường Xã Thị trấn', 'Tọa độ x',
       'Tọa độ y'],
      dtype='object')

## Preprocessing

In [9]:
def safe_mean_round(s):
    values = [x for x in s if pd.notna(x)]
    return round(float(np.nanmean(values)), 2) if values else None

In [10]:
# bỏ những dòng không có giá
df.drop_duplicates(inplace=True) 
df1 = df.dropna(subset=['Khoảng giá', 'Diện tích']).copy()
df1 = df1[(df1["Khoảng giá"] >= 100000000) | (df1["Loại quảng cáo"] != "Bán")]

In [11]:
df1["Căn góc"] = df1["Căn góc"].astype(str)

In [21]:
df1["Loại BĐS"].unique()

array(['căn hộ chung cư', 'đất', 'shophouse, nhà phố thương mại',
       'kho, nhà xưởng', 'kho, nhà xưởng, đất',
       'trang trại, khu nghỉ dưỡng', 'cửa hàng, ki ốt',
       'chung cư mini, căn hộ dịch vụ', 'văn phòng', 'nhà riêng',
       'nhà mặt phố', 'nhà biệt thự, liền kề', 'đất nền dự án',
       'nhà trọ, phòng trọ', 'condotel', 'loại bất động sản khác'],
      dtype=object)

In [12]:
group_cols = [ 
    'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận',  'Địa chỉ 1', 'Căn góc',
    'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án', 
    'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào',
]

agg_df = (
    df1
    .groupby(group_cols, dropna=False)
    .agg(
        mean_unique_khoang_gia = ('Khoảng giá', lambda s: np.mean(np.unique(s)) if len(np.unique(s))>0 else np.nan),
        phap_ly=("Pháp lý", "first"),
        noi_that=("Nội thất", "first"),
        dia_chi_2=("Địa chỉ 2", "first"),
        tieu_de=("Tiêu đề", "first"),
        Mo_ta=("Mô tả", "first"),
        Phuong=("Phường Xã Thị trấn", "first"),
        Toa_do_x=("Tọa độ x", "first"),
        Toa_do_y=("Tọa độ y", "first")
    )
    .reset_index()
)

# nếu muốn chuyển mean sang triệu hoặc tỷ (ví dụ sang triệu VND):
agg_df['mean_unique_khoang_gia_million'] = agg_df['mean_unique_khoang_gia'] / 1e6

In [13]:
agg_df.shape

(298381, 25)

In [14]:
# làm tròn chữ cái đầu tiên của tất cả các giá trị trong cột bds_type
agg_df['Loại BĐS'] = agg_df['Loại BĐS'].str.strip().str.capitalize()

# rename loại bds
mapping = {
    "Nhà biệt thự, liền kề": "Nhà biệt thự / Nhà liền kề",
    "Nhà mặt phố": "Nhà phố",
    "Nhà riêng": "Nhà ở",
    "Nhà trọ, phòng trọ": "Nhà trọ"
}
agg_df["Loại BĐS"] = agg_df["Loại BĐS"].replace(mapping)

# rename pháp lý
mask = agg_df["phap_ly"].notna() & (agg_df["phap_ly"] != "Hợp đồng mua bán")
agg_df.loc[mask, "phap_ly"] = "Sổ đỏ/ Sổ hồng"

# rename nội thất
col_lower = agg_df["noi_that"].str.lower()
agg_df["noi_that"] = np.select(
    [
        col_lower.str.contains("cơ bản", na=False),
        col_lower.str.contains("không nội thất", na=False),
        col_lower.str.contains("đầy đủ", na=False),
    ],
    [
        "cơ bản",
        "không nội thất",
        "đầy đủ",
    ],
    default=np.where(agg_df["noi_that"].notna(), "đầy đủ", "NaN")
)

agg_df = agg_df[agg_df["Loại BĐS"].isin(["Nhà phố", "Nhà ở", "Căn hộ chung cư", "Đất"])]


In [15]:
agg_df["so_tang"] = agg_df["Số tầng"].str.extract(r'(\d+)')
agg_df["so_tang"] = pd.to_numeric(agg_df["so_tang"], errors='coerce')

agg_df["mat_tien"] = agg_df["Mặt tiền"].str.extract(r'(\d+)')
agg_df["mat_tien"] = pd.to_numeric(agg_df["mat_tien"], errors='coerce')

agg_df["duong_vao"] = agg_df["Đường vào"].str.extract(r'(\d+)')
agg_df["duong_vao"] = pd.to_numeric(agg_df["duong_vao"], errors='coerce')

In [16]:
agg_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 236021 entries, 1759 to 287181
Data columns (total 28 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   Loại quảng cáo                  236021 non-null  object 
 1   Loại BĐS                        236021 non-null  object 
 2   Tỉnh thành phố                  236021 non-null  object 
 3   Quận                            236021 non-null  object 
 4   Địa chỉ 1                       236021 non-null  object 
 5   Căn góc                         236021 non-null  object 
 6   Diện tích                       236021 non-null  float64
 7   Số phòng ngủ                    140975 non-null  float64
 8   Số phòng tắm - vệ sinh          132602 non-null  float64
 9   Tên dự án                       59330 non-null   object 
 10  Hướng nhà                       70401 non-null   object 
 11  Hướng ban công                  42119 non-null   object 
 12  Số tầng           

## Loại bỏ nhiễu

In [17]:
ban_df = agg_df[agg_df["Loại quảng cáo"] == "Bán"]
thue_df = agg_df[agg_df["Loại quảng cáo"] == "Cho thuê"]

In [18]:
price_col = 'mean_unique_khoang_gia_million'
group_cols = ['Loại BĐS', 'Tỉnh thành phố', 'Quận']

# thu thập index các dòng cần drop
idx_to_drop = []
for _, g in ban_df.groupby(group_cols):
    n = len(g)
    # luôn bỏ ít nhất 1 dòng
    k = max(1, int(np.ceil(n*0.01)))
    if k > 0:
        # chọn k dòng có giá cao nhất trong nhóm
        idx_to_drop.extend(g.nlargest(k, price_col).index.tolist())

# dataframe sau khi đã loại bỏ 1% hàng top theo mỗi nhóm
ban_df = ban_df.drop(index=idx_to_drop).reset_index(drop=True)

In [19]:
# df2 = ban_df[(ban_df["Tỉnh thành phố"] == "Hà Nội") & (ban_df["Loại BĐS"] == "Nhà ở")]
df2 = ban_df[(ban_df["Tỉnh thành phố"] == "Hà Nội")]
# df2 = ban_df

In [20]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 55424 entries, 4870 to 170362
Data columns (total 28 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Loại quảng cáo                  55424 non-null  object 
 1   Loại BĐS                        55424 non-null  object 
 2   Tỉnh thành phố                  55424 non-null  object 
 3   Quận                            55424 non-null  object 
 4   Địa chỉ 1                       55424 non-null  object 
 5   Căn góc                         55424 non-null  object 
 6   Diện tích                       55424 non-null  float64
 7   Số phòng ngủ                    35400 non-null  float64
 8   Số phòng tắm - vệ sinh          32351 non-null  float64
 9   Tên dự án                       15241 non-null  object 
 10  Hướng nhà                       15039 non-null  object 
 11  Hướng ban công                  11656 non-null  object 
 12  Số tầng                         2

In [20]:
# Separate the features (X) and the target (y)
X = df2.drop(columns=["mean_unique_khoang_gia", 'mean_unique_khoang_gia_million', "Số tầng", "Mặt tiền", "Đường vào"])
y = df2['mean_unique_khoang_gia_million']

# Split the data into a test subset (smaller portion) and a training subset (larger portion)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
# X_test.to_csv("/lakehouse/default/Files/HaNoiXtestData.csv", index=False)
# y_test.to_csv("/lakehouse/default/Files/HaNoiYtestData.csv", index=False)

In [ ]:
X.columns

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

## Tree-based

In [ ]:
numerical_columns = X.select_dtypes(exclude = 'object').columns
categorical_columns = X_train.select_dtypes(include = 'object').columns
categorical_column_indices = [X_train.columns.get_loc(col) for col in categorical_columns]

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [ ]:
# Convert NaN values to strings in the specified categorical columns
for col_index in categorical_column_indices:
    X_train.iloc[:, col_index] = X_train.iloc[:, col_index].astype(str).fillna('NaN')
    X_test.iloc[:, col_index] = X_test.iloc[:, col_index].astype(str).fillna('NaN')

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [ ]:
# Additional regressors for models_with_categorical
models_with_categorical = [
    # DecisionTreeRegressor(),
    # RandomForestRegressor(max_depth=8, random_state=42),
    # RandomForestRegressor(),
    # GradientBoostingRegressor(learning_rate=0.03, max_depth=10, random_state=42),
    # XGBRegressor(learning_rate=0.03, max_depth=10, random_state=42),
    # XGBRegressor(),
    # SVR(),
    CatBoostRegressor(iterations=2995,  # Specify the number of boosting iterations
                      learning_rate=0.08222472905612799,  # Learning rate
                      depth = 11,
                      verbose=100,  # Print progress every 100 iterations
                      cat_features=categorical_column_indices,
                      l2_leaf_reg=4.36933961863642,
                      random_strength=3.2510914157888672e-09,
                      bagging_temperature=3.221967027448404
    )
    # CatBoostRegressor()  
]

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [ ]:
# # # Encode input
# # X_encoded = X.copy()
# # for feature in categorical_columns:
# #   label_encoder = LabelEncoder()
# #   X_encoded[feature] = label_encoder.fit_transform(X_encoded[feature])

# # X_train_encoded, X_test_encoded, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
# results_df = pd.DataFrame()

# for model in models_with_categorical:
#     # Fit the model
#     model.fit(X_train, y_train)

#     # Predict on the test set
#     y_pred = model.predict(X_test)

#     # Calculate metrics
#     mse = mean_squared_error(y_test, y_pred)
#     r2 = r2_score(y_test, y_pred)

#     # Append results to the dataframe
#     results_df = pd.concat([
#         results_df,
#         pd.DataFrame([{
#             'Model': type(model).__name__,
#             'MSE': mse,
#             'R2_Score': r2
#         }])
#     ], ignore_index=True)

    
#     df_eval = X_test[["Loại BĐS"]].copy()
#     df_eval["y_true"] = y_test.values
#     df_eval["y_pred"] = y_pred

#     r2_by_bds_type = (
#         df_eval
#         .groupby("Loại BĐS")
#         .apply(lambda x: r2_score(x["y_true"], x["y_pred"]))
#     )

#     print(r2_by_bds_type)

# # # Display the results dataframe
# # results_df

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [ ]:
# df_eval = X_test[["Loại BĐS"]].copy()
# df_eval["y_true"] = y_test.values
# df_eval["y_pred"] = y_pred

# r2_by_bds_type = (
#     df_eval
#     .groupby("Loại BĐS")
#     .apply(lambda x: r2_score(x["y_true"], x["y_pred"]))
# )

# print(r2_by_bds_type)

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [ ]:
# # Tránh chia cho 0
# mask = y_test != 0
# mape = (
#     np.mean(
#         np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])
#     ) * 100
# )

# results_df = pd.concat([
#     results_df,
#     pd.DataFrame([{
#         'Model': type(model).__name__,
#         'MSE': mse,
#         'R2_Score': r2,
#         'MAPE_%': mape
#     }])
# ], ignore_index=True)
# print(results_df)

# mape_by_bds_type = (
#     df_eval[df_eval["y_true"] != 0]
#     .groupby("Loại BĐS")
#     .apply(lambda x: np.mean(
#         np.abs((x["y_true"] - x["y_pred"]) / x["y_true"])
#     ) * 100)
# )
# print(mape_by_bds_type)

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [ ]:
# test_cases = pd.DataFrame([{
#     'Loại quảng cáo': 'Bán',
#     'Loại BĐS': 'Căn hộ chung cư',
#     'Tỉnh thành phố': 'Hà Nội',
#     'Quận': 'Hoàn Kiếm',
#     'Địa chỉ 1': 'phố Hai Bà Trưng',
#     'Diện tích': 112,
#     'Số phòng ngủ': np.nan,
#     'Số phòng tắm - vệ sinh': np.nan,
#     'Tên dự án': 'NaN',
#     'Hướng nhà': 'Đông - Bắc',
#     'Hướng ban công': 'Đông - Bắc',
#     'phap_ly': 'Sổ đỏ/ Sổ hồng',
#     'noi_that': 'NaN',
#     'so_tang': 5,
#     'mat_tien': 4.3,
#     'duong_vao': 40
# }])

# model.predict(test_cases)

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [ ]:
# r2_by_province

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [87]:
import optuna

def objective(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "depth": trial.suggest_int("depth", 4, 16),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.5, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 5),
        "loss_function": "MAE",
        "verbose": 0,
    }

    model = CatBoostRegressor(**params)

    model.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),
        cat_features=categorical_column_indices,
        early_stopping_rounds=100,
        verbose=False
    )

    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    print(f"Trial {trial.number} | RMSE: {rmse:.4f} | R2: {r2:.4f}")
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    print(f"MAPE: {mape:.2f}%")

    return rmse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, 92, Finished, Available, Finished, False)

[I 2026-04-28 16:20:39,666] A new study created in memory with name: no-name-b27979fd-06c4-4804-b901-5329df1455a2
[I 2026-04-28 16:47:42,951] Trial 18 finished with value: 17183.953042203393 and parameters: {'iterations': 2409, 'depth': 7, 'learning_rate': 0.18981386759264246, 'l2_leaf_reg': 8.826908153421487, 'random_strength': 0.5899236694023378, 'bagging_temperature': 3.4336615211668238}. Best is trial 12 with value: 15595.844328986075.


Trial 0 | RMSE: 44338.1563 | R2: 0.2062
MAPE: 42.37%
Trial 1 | RMSE: 32664.7085 | R2: 0.5692
MAPE: 28.17%
Trial 2 | RMSE: 28807.1594 | R2: 0.6649
MAPE: 26.23%
Trial 3 | RMSE: 40361.8347 | R2: 0.3422
MAPE: 32.32%
Trial 4 | RMSE: 38851.0687 | R2: 0.3905
MAPE: 30.81%
Trial 5 | RMSE: 20537.8877 | R2: 0.8297
MAPE: 24.94%
Trial 6 | RMSE: 28034.9368 | R2: 0.6826
MAPE: 26.00%
Trial 7 | RMSE: 40677.5833 | R2: 0.3319
MAPE: 31.68%
Trial 8 | RMSE: 17138.0589 | R2: 0.8814
MAPE: 27.20%
Trial 9 | RMSE: 20325.6615 | R2: 0.8332
MAPE: 26.06%
Trial 10 | RMSE: 16946.3448 | R2: 0.8840
MAPE: 27.66%
Trial 11 | RMSE: 18450.3074 | R2: 0.8625
MAPE: 26.75%
Trial 12 | RMSE: 15595.8443 | R2: 0.9018
MAPE: 27.89%
Trial 13 | RMSE: 17298.2786 | R2: 0.8792
MAPE: 25.14%
Trial 14 | RMSE: 18244.0620 | R2: 0.8656
MAPE: 27.39%
Trial 15 | RMSE: 17498.2864 | R2: 0.8764
MAPE: 35.62%
Trial 16 | RMSE: 19026.7742 | R2: 0.8538
MAPE: 28.97%
Trial 17 | RMSE: 18991.1877 | R2: 0.8544
MAPE: 26.92%
Trial 19 | RMSE: 24167.7650 | R2: 0.76

In [ ]:
# model = CatBoostRegressor(iterations=1000,  # Specify the number of boosting iterations
#                           learning_rate=0.1,  # Learning rate
#                           depth = 16,
#                           verbose=100,  # Print progress every 100 iterations
#                           random_seed=42,
#                           cat_features=categorical_column_indices)

# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

# r2_score(y_test, y_pred)

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [ ]:

# # Get feature importances
# feature_importance = model.get_feature_importance()

# # Get feature names
# feature_names = X_train.columns  # Replace with the actual column names of your features

# # Create a dictionary to store feature names and their corresponding importance scores
# feature_importance_dict = dict(zip(feature_names, feature_importance))

# # Sort feature importances and select the top 10
# top_features = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)[:10]

# # Print the top 10 most important features
# print("Top 10 Most Important Features:")
# for feature, importance in top_features:
#     print(f"{feature}: {importance}")

# # Plot only the top 10 feature importances in descending order
# top_feature_names, top_feature_importance = zip(*top_features)

# plt.barh(top_feature_names[::-1], top_feature_importance[::-1])  # Reverse the order for descending plot
# plt.xlabel('Feature Importance Score')
# plt.ylabel('Features')
# plt.title('Top 10 Most Important Features')
# plt.show()
# ## Randomly select 4000, 5000, 6000 samples from the dataset
# ### 2000 samples
# def training(df):

#     # Separate the features (X) and the target (y)
#     X = df.drop(columns=['ad_price'])
#     y = df['ad_price']

#     # Split the data into a test subset (smaller portion) and a training subset (larger portion)
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#     numerical_columns = X.select_dtypes(exclude = 'object').columns
#     categorical_columns = X_train.select_dtypes(include = 'object').columns
#     categorical_column_indices = [X_train.columns.get_loc(col) for col in categorical_columns]

#     # Convert NaN values to strings in the specified categorical columns
#     for col_index in categorical_column_indices:
#         X_train.iloc[:, col_index] = X_train.iloc[:, col_index].astype(str).fillna('NaN')
#         X_test.iloc[:, col_index] = X_test.iloc[:, col_index].astype(str).fillna('NaN')

#     # Initialize an empty dataframe to store results
#     results_df = pd.DataFrame(columns=['Model', 'MSE', 'R2_Score'])

#     # Additional regressors for models_with_categorical
#     models_with_categorical = [
#         DecisionTreeRegressor(),
#         RandomForestRegressor(max_depth=7, random_state=42),
#         GradientBoostingRegressor(),
#         XGBRegressor(learning_rate=0.2, max_depth=5, random_state=42),
#         SVR(),
#         CatBoostRegressor(iterations=1000,  # Specify the number of boosting iterations
#                           learning_rate=0.1,  # Learning rate
#                           depth = 7,
#                           verbose=100,  # Print progress every 100 iterations
#                           random_seed=42,
#                           cat_features=categorical_column_indices)
#     ]

#     # Additional regressors for models_without_categorical
#     models_without_categorical = [
#         LinearRegression(),
#         Ridge(),
#         Lasso(),
#         KNeighborsRegressor()
#     ]
#     for model in models_without_categorical:
#             # Fit the model
#             model.fit(X_train[numerical_columns].fillna(X[numerical_columns].mean()), y_train)

#             # Predict on the test set
#             y_pred = model.predict(X_test[numerical_columns].fillna(X[numerical_columns].mean()))

#             # Calculate metrics
#             mse = mean_squared_error(y_test, y_pred)
#             r2 = r2_score(y_test, y_pred)

#             # Append results to the dataframe
#             results_df = results_df.append({'Model': type(model).__name__,
#                                             'MSE': mse,
#                                             'R2_Score': r2},
#                                           ignore_index=True)
#     # Encode input
#     X_encoded = X.copy()
#     for feature in categorical_columns:
#       label_encoder = LabelEncoder()
#       X_encoded[feature] = label_encoder.fit_transform(X_encoded[feature])

#     X_train_encoded, X_test_encoded, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

#     for model in models_with_categorical:
#             # Fit the model
#             model.fit(X_train_encoded.fillna(X_encoded.mean()), y_train)

#             # Predict on the test set
#             y_pred = model.predict(X_test_encoded.fillna(X_encoded.mean()))

#             # Calculate metrics
#             mse = mean_squared_error(y_test, y_pred)
#             r2 = r2_score(y_test, y_pred)

#             # Append results to the dataframe
#             results_df = results_df.append({'Model': type(model).__name__,
#                                             'MSE': mse,
#                                             'R2_Score': r2},
#                                           ignore_index=True)
#     # Display the results dataframe
#     print(results_df)
# df_2000 = df2.sample(2000)
# training(df_2000)
# ### 5000 samples
# df_5000 = df2.sample(5000)
# training(df_5000)
# ### 8000 samples
# df_8000 = df2.sample(8000)
# training(df_8000)

StatementMeta(, ed6222fb-e825-4b36-89f6-19f81cd10387, -1, Cancelled, , Cancelled, True)

In [1]:
!pip install tqdm category_encoders underthesea optuna scikit-learn -qq

"""
Research V10: Flexible Deduplication (Title-based) + Top 1% Drop.
Aims to keep more data variance while removing extreme outliers.
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import xgboost as xgb
import lightgbm as lgb
import category_encoders as ce
import joblib
from sklearn.cluster import MiniBatchKMeans
from underthesea import text_normalize
import optuna
optuna.logging.set_verbosity(optuna.logging.INFO) # Hiện log từng trial để theo dõi progress

METRO_STATIONS = [(21.028, 105.828), (21.015, 105.820), (21.015, 105.810), (21.030, 105.800), (21.002, 105.815), (20.975, 105.776)]
CENTER_LAT, CENTER_LON = 21.0285, 105.8542

def get_dist_to_metro(px, py):
    return min([np.sqrt((px-mx)**2 + (py-my)**2) for mx, my in METRO_STATIONS])

def load_and_clean_v10():
    print("Loading data and applying V10 Strategy (Flexible Dedup)...")
    df = pd.read_csv('/lakehouse/default/Files/filtered_hanoi_data.csv', encoding='utf-8', low_memory=False)
    
    top_4_types = ['căn hộ chung cư', 'nhà riêng', 'đất', 'nhà mặt phố']
    df = df[(df['Loại quảng cáo'] == 'Bán') & (df['Loại BĐS'].isin(top_4_types))]
    
    for col in ['Khoảng giá', 'Diện tích', 'Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Mặt tiền', 'Đường vào', 'Tọa độ x', 'Tọa độ y']:
        # df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0]
        df[col] = pd.to_numeric(df[col], errors='coerce')
 
    df = df[df['Khoảng giá'] >= 1e8].dropna(subset=['Khoảng giá', 'Diện tích'])
    
    # Deduplication theo đúng yêu cầu (15 cột cấu trúc + mean of unique prices)
    group_cols = [
        'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1', 'Căn góc',
        'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án',
        'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào',
    ]
    
    df_dedup = (
        df.groupby(group_cols, dropna=False)
        .agg(
            mean_unique_khoang_gia=('Khoảng giá', lambda s: np.mean(np.unique(s[s.notna()])) if len(np.unique(s[s.notna()])) > 0 else np.nan),
            phap_ly=('Pháp lý', 'first'),
            noi_that=('Nội thất', 'first'),
            dia_chi_2=('Địa chỉ 2', 'first'),
            tieu_de=('Tiêu đề', 'first'),
            Mo_ta=('Mô tả', 'first'),
            Phuong=('Phường Xã Thị trấn', 'first'),
            Toa_do_x=('Tọa độ x', 'first'),
            Toa_do_y=('Tọa độ y', 'first'),
        )
        .reset_index()
        .rename(columns={
            'mean_unique_khoang_gia': 'Price',
            'Mo_ta': 'Mô tả', 'Phuong': 'Phường Xã Thị trấn',
            'Toa_do_x': 'Tọa độ x', 'Toa_do_y': 'Tọa độ y',
            'dia_chi_2': 'Địa chỉ 2', 'tieu_de': 'Tiêu đề',
            'phap_ly': 'Pháp lý', 'noi_that': 'Nội thất'
        })
    )
    df_dedup = df_dedup.dropna(subset=['Price'])
    print(f"Rows after Deduplication: {len(df_dedup)}")
    
    # Top 1% Drop per (Type, District)
    idx_to_drop = []
    for _, g in df_dedup.groupby(['Loại BĐS', 'Quận']):
        k = max(1, int(np.ceil(len(g) * 0.01)))
        idx_to_drop.extend(g.nlargest(k, 'Price').index.tolist())
    
    df_final = df_dedup.drop(index=idx_to_drop).reset_index(drop=True)
    print(f"Final rows after Top 1% Drop: {len(df_final)}")
    return df_final

def extract_features(df):
    print("Extracting features...")
    df['clean_desc'] = df['Mô tả'].astype(str).apply(text_normalize).str.lower()
    desc = df['clean_desc']
    
    # NLP Features
    df['feat_oto'] = desc.str.contains('xe hơi|ô tô|oto').astype(int)
    df['feat_tranh'] = desc.str.contains('tránh').astype(int)
    df['feat_no_hau'] = desc.str.contains('nở hậu').astype(int)
    df['feat_thang_may'] = desc.str.contains('thang máy').astype(int)
    df['feat_kinh_doanh'] = desc.str.contains('kinh doanh|buôn bán').astype(int)
    df['feat_mat_tien'] = desc.str.contains('mặt phố|mặt đường').astype(int)
    df['feat_noi_that'] = desc.str.contains('nội thất|đầy đủ|tiện nghi').astype(int)
    df['feat_so_do'] = desc.str.contains('sổ đỏ|sổ hồng').astype(int)
    df['feat_chinh_chu'] = desc.str.contains('chính chủ').astype(int)
    
    df['dist_to_metro'] = df.apply(lambda r: get_dist_to_metro(r['Tọa độ x'], r['Tọa độ y']), axis=1)
    df['dist_to_center'] = np.sqrt((df['Tọa độ x'] - CENTER_LAT)**2 + (df['Tọa độ y'] - CENTER_LON)**2)
    df['type_dist'] = df['Loại BĐS'].astype(str) + "_" + df['Quận'].astype(str)
    
    # Số phòng tắm dùng từ group_cols, cần rename về tên chuẩn nếu chưa có
    for col in ['Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Mặt tiền', 'Đường vào']:
        df[col] = df[col].fillna(df.groupby('Loại BĐS')[col].transform('median')).fillna(0)
    
    # Căn góc
    df['can_goc'] = (df['Căn góc'].astype(str).str.lower().isin(['có', 'yes', '1', 'true', 'căn góc']))
    
    df['dien_tich_per_tang'] = df['Diện tích'] / df['Số tầng'].replace(0, 1)
    df['mat_tien_x_tang'] = df['Mặt tiền'] * df['Số tầng']
    return df

ALL_FEATURES = ['Loại BĐS', 'Quận', 'Phường Xã Thị trấn', 'Địa chỉ 1', 'Diện tích', 'Tọa độ x', 'Tọa độ y',
                'Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Mặt tiền', 'Đường vào',
                'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất',
                'can_goc',
                'feat_kinh_doanh', 'feat_mat_tien', 'dist_to_metro', 'dist_to_center',
                'type_dist', 'loc_cluster',
                'feat_oto', 'feat_tranh', 'feat_no_hau', 'feat_thang_may',
                'feat_noi_that', 'feat_so_do', 'feat_chinh_chu',
                'dien_tich_per_tang', 'mat_tien_x_tang']
CATEGORICAL = ['Loại BĐS', 'Quận', 'Địa chỉ 1', 'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất', 'type_dist', 'loc_cluster', 'Phường Xã Thị trấn']

if __name__ == "__main__":
    df = load_and_clean_v10()
    df = extract_features(df)
    
    kmeans = MiniBatchKMeans(n_clusters=400, random_state=42, n_init=3)
    coords = df[['Tọa độ x', 'Tọa độ y']].copy()
    coords['Tọa độ x'] = coords['Tọa độ x'].fillna(coords['Tọa độ x'].median())
    coords['Tọa độ y'] = coords['Tọa độ y'].fillna(coords['Tọa độ y'].median())
    df['loc_cluster'] = kmeans.fit_predict(coords)
    
    X = df[ALL_FEATURES]
    y = np.log1p(df['Price'])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    
    encoder = ce.TargetEncoder(cols=CATEGORICAL)
    X_train_enc = encoder.fit_transform(X_train, y_train)
    X_test_enc = encoder.transform(X_test)
    
    print("\nPhase 1: Optuna tuning XGBoost...")
    def objective_xgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 2000, 5000),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.03),
            'max_depth': trial.suggest_int('max_depth', 10, 16),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'tree_method': 'hist'  # XGBoost: CPU (AMD GPU không hỗ trợ CUDA)
        }
        m = xgb.XGBRegressor(**params)
        m.fit(X_train_enc, y_train)
        pred = np.expm1(m.predict(X_test_enc))
        return mean_absolute_percentage_error(np.expm1(y_test), pred)
    
    study_xgb = optuna.create_study(direction='minimize')
    study_xgb.optimize(objective_xgb, n_trials=20, show_progress_bar=True)
    
    print("\nPhase 2: Optuna tuning LightGBM...")
    def objective_lgb(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 2000, 5000),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.03),
            'num_leaves': trial.suggest_int('num_leaves', 255, 1023),
            'verbose': -1
        }
        m = lgb.LGBMRegressor(**params)
        m.fit(X_train_enc, y_train)
        pred = np.expm1(m.predict(X_test_enc))
        return mean_absolute_percentage_error(np.expm1(y_test), pred)
    
    study_lgb = optuna.create_study(direction='minimize')
    study_lgb.optimize(objective_lgb, n_trials=20, show_progress_bar=True)
    
    print("\nPhase 3: Final training and Ensemble...")
    m_xgb = xgb.XGBRegressor(**study_xgb.best_params)
    m_xgb.fit(X_train_enc, y_train)
    m_lgb = lgb.LGBMRegressor(**study_lgb.best_params)
    m_lgb.fit(X_train_enc, y_train)
    
    p_xgb = np.expm1(m_xgb.predict(X_test_enc))
    p_lgb = np.expm1(m_lgb.predict(X_test_enc))
    y_true = np.expm1(y_test)
    
    best_w, best_mape = 0.5, 999
    for w in np.arange(0.0, 1.01, 0.05):
        p = w * p_xgb + (1 - w) * p_lgb
        m = mean_absolute_percentage_error(y_true, p)
        if m < best_mape:
            best_mape, best_w = m, w
    
    print(f"\nFINAL V10 MAPE: {best_mape*100:.2f}% (XGB Weight: {best_w:.2f})")
    
    # joblib.dump(m_xgb, '/lakehouse/default/Files/master_xgb_v10.joblib')
    # joblib.dump(m_lgb, '/lakehouse/default/Files/master_lgb_v10.joblib')
    # joblib.dump(encoder, '/lakehouse/default/Files/master_encoder_v10.joblib')
    # joblib.dump(kmeans, '/lakehouse/default/Files/master_kmeans_v10.joblib')
    # print("Models V10 saved.")
    
    # --- Test trên HaNoiXtestData & HaNoiYtestData ---
    print("\n--- Testing on HaNoiData (External) ---")
    try:
        X_ext = pd.read_csv('/lakehouse/default/Files/HaNoiXtestData.csv', encoding='utf-8')
        y_ext = pd.read_csv('/lakehouse/default/Files/HaNoiYtestData.csv')
        
        # Rename garbled columns to match our pipeline
        col_idx = X_ext.columns.tolist()
        rename_map = {}
        if len(col_idx) > 1: rename_map[col_idx[1]] = 'Loại BĐS'
        if len(col_idx) > 3: rename_map[col_idx[3]] = 'Quận'
        if len(col_idx) > 6: rename_map[col_idx[6]] = 'Diện tích'
        if len(col_idx) > 7: rename_map[col_idx[7]] = 'Số phòng ngủ'
        X_ext = X_ext.rename(columns={**rename_map,
            'Phuong': 'Phường Xã Thị trấn', 'dia_chi_2': 'Địa chỉ 2',
            'Mo_ta': 'Mô tả', 'Toa_do_x': 'Tọa độ x', 'Toa_do_y': 'Tọa độ y',
            'so_tang': 'Số tầng', 'mat_tien': 'Mặt tiền', 'duong_vao': 'Đường vào'
        })
        
        # Ensure all required columns exist
        for c in ['Địa chỉ 1', 'Căn góc', 'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất', 
                  'Số phòng tắm - vệ sinh', 'Phường Xã Thị trấn']:
            if c not in X_ext.columns:
                X_ext[c] = np.nan
        
        for col in ['Diện tích', 'Tọa độ x', 'Tọa độ y', 'Số tầng', 'Mặt tiền', 'Đường vào', 'Số phòng ngủ']:
            X_ext[col] = pd.to_numeric(X_ext[col], errors='coerce').fillna(0)
        
        X_ext = extract_features(X_ext.copy())
        X_ext['loc_cluster'] = kmeans.predict(X_ext[['Tọa độ x', 'Tọa độ y']].fillna(X_ext[['Tọa độ x', 'Tọa độ y']].median()))
        
        X_ext_enc = encoder.transform(X_ext[ALL_FEATURES])
        p_xgb_ext = np.expm1(m_xgb.predict(X_ext_enc))
        p_lgb_ext = np.expm1(m_lgb.predict(X_ext_enc))
        y_pred_ext = best_w * p_xgb_ext + (1 - best_w) * p_lgb_ext
        
        y_true_ext = y_ext['mean_unique_khoang_gia_million'] * 1e6
        mape_ext = mean_absolute_percentage_error(y_true_ext, y_pred_ext)
        print(f"MAPE on HaNoiData (External): {mape_ext*100:.2f}%")
    except Exception as e:
        print(f"External test failed: {e}")



Phase 1: Optuna tuning XGBoost...


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-05-09 15:50:08,506] Trial 19 finished with value: 0.1738007744573536 and parameters: {'n_estimators': 2879, 'learning_rate': 0.02148856543641319, 'max_depth': 11, 'subsample': 0.8437122524586141}. Best is trial 13 with value: 0.17296909278473768.

Phase 2: Optuna tuning LightGBM...


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-05-09 15:51:52,327] Trial 0 finished with value: 0.1759138686676124 and parameters: {'n_estimators': 3069, 'learning_rate': 0.008403503401575621, 'num_leaves': 679}. Best is trial 0 with value: 0.1759138686676124.
[I 2026-05-09 15:53:02,003] Trial 1 finished with value: 0.17626046677132784 and parameters: {'n_estimators': 4291, 'learning_rate': 0.008812044761261382, 'num_leaves': 304}. Best is trial 0 with value: 0.1759138686676124.
[I 2026-05-09 15:54:11,376] Trial 2 finished with value: 0.1759400439341949 and parameters: {'n_estimators': 3232, 'learning_rate': 0.02182704964751168, 'num_leaves': 446}. Best is trial 0 with value: 0.1759138686676124.
[I 2026-05-09 15:55:30,365] Trial 3 finished with value: 0.1788084398751759 and parameters: {'n_estimators': 2034, 'learning_rate': 0.028800916629900367, 'num_leaves': 836}. Best is trial 0 with value: 0.1759138686676124.
[I 2026-05-09 15:58:05,834] Trial 4 finished with value: 0.1794789406702085 and parameters: {'n_estimators': 338

In [2]:
joblib.dump(m_xgb, '/lakehouse/default/Files/master_xgb_v10.joblib')
joblib.dump(m_lgb, '/lakehouse/default/Files/master_lgb_v10.joblib')
joblib.dump(encoder, '/lakehouse/default/Files/master_encoder_v10.joblib')
joblib.dump(kmeans, '/lakehouse/default/Files/master_kmeans_v10.joblib')

['/lakehouse/default/Files/master_kmeans_v10.joblib']

In [6]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55224 entries, 0 to 55223
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Loại BĐS                55224 non-null  object 
 1   Quận                    55224 non-null  object 
 2   Phường Xã Thị trấn      8116 non-null   object 
 3   Địa chỉ 1               55224 non-null  object 
 4   Diện tích               55224 non-null  float64
 5   Tọa độ x                55218 non-null  float64
 6   Tọa độ y                55218 non-null  float64
 7   Số tầng                 55224 non-null  float64
 8   Số phòng ngủ            55224 non-null  float64
 9   Số phòng tắm - vệ sinh  55224 non-null  float64
 10  Mặt tiền                55224 non-null  float64
 11  Đường vào               55224 non-null  float64
 12  Hướng nhà               15003 non-null  object 
 13  Hướng ban công          11654 non-null  object 
 14  Pháp lý                 49031 non-null